# Flight Emissions Data

We get flight data from OpenSky. We merge that with Europa data about emissions intensity by aircraft model.


## Imports and Constants

In [34]:
import datetime as dt
import zoneinfo
import json
# import json
# from pprint import pprint
# import urllib.request 
import os
# from statistics import median, mean
import logging

from tqdm import tqdm
import polars as pl
from opensky_api import OpenSkyApi
import numpy as np

In [4]:

# will be created if doesn't exist
data_dir = "data"

# from here: https://opensky-network.org/datasets/#metadata/aircraftDatabase
mapping_csv = os.path.join(data_dir, "raw/aircraftDatabase.csv.gz")

# from here: https://www.eea.europa.eu/publications/emep-eea-guidebook-2023/part-b-sectoral-guidance-chapters/1-energy/1-a-combustion/1-a-3-a-aviation.3/view
# https://www.eea.europa.eu/publications/emep-eea-guidebook-2023/part-b-sectoral-guidance-chapters/1-energy/1-a-combustion/1-a-3-a-aviation.3/at_download/file
emissions_spreadsheet = os.path.join(data_dir, "raw/1.A.3.a Aviation -Annex 1 - Master emissions calculator - 2023 - Protected - v1.5_18_09_2024.xlsx")

# downloaded from
# https://data.cso.ie/
# discovered from Wikipedia: https://en.wikipedia.org/wiki/Dublin_Airport#Statistics
passenger_numbers_sheet = os.path.join(data_dir, "popular_destinations.csv")

opensky_api_secret_file = "/home/matthew/.local/share/credentials/opensky_api_key.json"

results_dir = os.path.join(data_dir, "results")
emissions_results_path = os.path.join(results_dir, "emissions.parquet")
planes_results_path = os.path.join(results_dir, "planes.parquet")

In [5]:
# microseconds per second
us_per_s = 1e6

# Melbourne and Sydney are the same timezone
tz_name = 'Australia/Sydney' 

In [6]:
airport_info = {
    # https://en.wikipedia.org/wiki/Melbourne_Airport
    "Melbourne": {
        "IATA": "MEL",
        "ICAO": "YMML",
        "WMO": 94866,
        "location": {
            'latitude': -37.673333, 
            'longitude': 144.843333,
        }
    },
    # https://en.wikipedia.org/wiki/Sydney_Airport
    "Sydney": {
        "IATA": "SYD",
        "ICAO": "YSSY",
        "WMO": 94767,
        "location": {
            'latitude': -33.946111, 
            'longitude': 151.177222
        }
    }
}

# arbitrarily choose Sydney as the airport
# look at all flights to/from Sydney from/to anywhere
# when later filter by from/to Melbourne
airport_id_type = "ICAO"
airport_id = airport_info['Sydney'][airport_id_type]
other_airport_id = airport_info['Melbourne'][airport_id_type]


In [7]:
sydney_tz = zoneinfo.ZoneInfo('Australia/Sydney')

## Download flight data from Open Sky

The OpenSky API is documented [here](https://openskynetwork.github.io/opensky-api/python.html#opensky_api.FlightData).


In [6]:
logger = logging.getLogger("opensky_api")
#logger.addHandler(logging.StreamHandler())
logger.setLevel(logging.DEBUG)

In [7]:
api = OpenSkyApi(client_json_path=opensky_api_secret_file)

In [8]:
_date = dt.date.today() - dt.timedelta(days=100)


# must include no more than 2 days in the timespan
# time_start = dt.datetime(2025, 10, 10, 0, 0, tzinfo=sydney_tz)
# time_end = dt.datetime(2025, 10, 15, 23, 59, 59, tzinfo=sydney_tz)
# time_start = dt.datetime.combine(_date, dt.time.min, tzinfo=sydney_tz)
# time_end = dt.datetime.combine(_date, dt.time.max, tzinfo=sydney_tz)

# time_start = dt.datetime.combine(_date, dt.time.min)
# time_end = dt.datetime.combine(_date, dt.time.max)

def get_epoch(t: dt.datetime) -> int:
    return round(t.timestamp())

In [9]:
today = dt.date.today()
_dates = [today - dt.timedelta(days=d) for d in range(95, 100)]
_dates

[datetime.date(2025, 10, 22),
 datetime.date(2025, 10, 21),
 datetime.date(2025, 10, 20),
 datetime.date(2025, 10, 19),
 datetime.date(2025, 10, 18)]

In [10]:
responses = []

for _date in tqdm(_dates):
    start_epoch = get_epoch(dt.datetime.combine(_date, dt.time.min))
    end_epoch = get_epoch(dt.datetime.combine(_date, dt.time.max))

    for airport in [airport_id, other_airport_id]:
        responses.extend(
            api.get_departures_by_airport(
                airport=airport, begin=start_epoch, end=end_epoch
            )
        )
        # responses.extend(
        #     api.get_arrivals_by_airport(
        #         airport=airport, begin=start_epoch, end=end_epoch
        #     )
        # )

100%|████████████████████████████████████████████████████████| 5/5 [00:07<00:00,  1.52s/it]


In [11]:
flight_api_data = pl.LazyFrame([f.__dict__ for f in responses])
flight_api_data.sink_csv(os.path.join(data_dir, "api_raw.csv"))
flight_api_data.sink_parquet(os.path.join(data_dir, "api_raw.parquet"))

## Process Flight Data

You can skip the previous section and start from here to read cached data from disk.

Parse datetimes, calculate duration. (e.g. `firstSeen` actually is departure time.)
We delete in-progress flights, because that duration will be wrong.

In [8]:
flights = (
    pl.scan_parquet(os.path.join(data_dir, "api_raw.parquet"))
    .filter(pl.col("estArrivalAirport").is_not_null())
    .filter(pl.col("estArrivalAirport") != pl.col("estDepartureAirport"))
    .with_columns(
        (pl.col("firstSeen") * us_per_s).cast(
            pl.Datetime()
        ).dt.convert_time_zone("Australia/Sydney"),
        (pl.col("lastSeen") * us_per_s).cast(
            pl.Datetime()
        ).dt.convert_time_zone("Australia/Sydney"),
    )
    .with_columns(
        (pl.col("lastSeen") - pl.col("firstSeen")).alias("flightDurationActual")
    )
    .select(
        pl.col("icao24").alias("ICAO24_HEX"),
        pl.col("firstSeen").alias("DEPARTURE_TIME"),
        pl.col("lastSeen").alias("ARRIVAL_TIME"),
        pl.col("flightDurationActual").alias("FLIGHT_DURATION_ACTUAL"),
        pl.col("estDepartureAirport").alias("DEPARTURE_AIRPORT"),
        pl.col("estArrivalAirport").alias("ARRIVAL_AIRPORT"),
    )
)
flights.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str
"""7c5307""",2025-10-23 08:56:18 AEDT,2025-10-23 09:47:20 AEDT,51m 2s,"""YSSY""","""YSCH"""
"""7c6b35""",2025-10-23 08:55:52 AEDT,2025-10-23 09:59:52 AEDT,1h 4m,"""YSSY""","""YRSY"""
"""7c6de2""",2025-10-23 08:49:20 AEDT,2025-10-23 09:59:43 AEDT,1h 10m 23s,"""YSSY""","""YMML"""
"""7c6d33""",2025-10-23 08:47:50 AEDT,2025-10-23 09:57:31 AEDT,1h 9m 41s,"""YSSY""","""YMML"""
"""7c6d2f""",2025-10-23 08:44:00 AEDT,2025-10-23 09:48:42 AEDT,1h 4m 42s,"""YSSY""","""YBBN"""


In [9]:
(
    flights
    .select(
        pl.col("ARRIVAL_TIME").min().alias("ARRIVAL_MIN"),
        pl.col("ARRIVAL_TIME").max().alias("ARRIVAL_MAX"),
        pl.col("DEPARTURE_TIME").min().alias("DEPARTURE_MIN"),
        pl.col("DEPARTURE_TIME").max().alias("DEPARTURE_MAX"),
    )
    .collect()
)

ARRIVAL_MIN,ARRIVAL_MAX,DEPARTURE_MIN,DEPARTURE_MAX
"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]"
2025-10-18 09:37:06 AEDT,2025-10-23 09:59:52 AEDT,2025-10-18 09:00:49 AEDT,2025-10-23 08:56:18 AEDT


In [10]:
# filter to flights between Melbourne and Sydney
# and during the time window (API might return extra)
flights = (
    flights
    .filter(
        pl.any_horizontal(
            (pl.col("DEPARTURE_AIRPORT") == pl.lit(airport_id))
            &
            (pl.col("ARRIVAL_AIRPORT") == pl.lit(other_airport_id)),
            (pl.col("DEPARTURE_AIRPORT") == pl.lit(other_airport_id))
            &
            (pl.col("ARRIVAL_AIRPORT") == pl.lit(airport_id))
        )
    )
    # .filter(pl.col("DEPARTURE_TIME") >= time_start)
    # .filter(pl.col("ARRIVAL_TIME") >= time_end)
)
flights.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str
"""7c6de2""",2025-10-23 08:49:20 AEDT,2025-10-23 09:59:43 AEDT,1h 10m 23s,"""YSSY""","""YMML"""
"""7c6d33""",2025-10-23 08:47:50 AEDT,2025-10-23 09:57:31 AEDT,1h 9m 41s,"""YSSY""","""YMML"""
"""7c1476""",2025-10-23 08:38:39 AEDT,2025-10-23 09:51:40 AEDT,1h 13m 1s,"""YSSY""","""YMML"""
"""7c7cce""",2025-10-23 08:27:15 AEDT,2025-10-23 09:42:23 AEDT,1h 15m 8s,"""YSSY""","""YMML"""
"""7c7a48""",2025-10-23 08:15:30 AEDT,2025-10-23 09:33:45 AEDT,1h 18m 15s,"""YSSY""","""YMML"""


In [11]:
flights.select("DEPARTURE_AIRPORT", "ARRIVAL_AIRPORT").unique().head().collect()

DEPARTURE_AIRPORT,ARRIVAL_AIRPORT
str,str
"""YSSY""","""YMML"""
"""YMML""","""YSSY"""


## Join to emissions data

We download a mapping file from OpenSky, to map icao24 IDs to flight model.
Browse the files [here](https://opensky-network.org/datasets/#metadata/aircraftDatabase).
With that page, `aircraftDatabase.csv` came from [here]("https://s3.opensky-network.org/data-samples/metadata/aircraftDatabase.csv").

Within this CSV:

* `icao24` matches the OpenSki API
* `typecode` matches what's in the emissions spreadsheet as `ICAO_24`

In [12]:
def download_file(url, path):
    dir_path = os.path.dirname(path)
    os.makedirs(dir_path, exist_ok=True)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)

In [13]:
# https://opensky-network.org/datasets/#metadata/aircraftDatabase
url = "https://s3.opensky-network.org/data-samples/metadata/aircraftDatabase.csv"
download_file(url, mapping_csv)

In [14]:
mapping_df = (
    pl.scan_csv(mapping_csv, skip_rows_after_header=1)
    .select(
        pl.col("icao24").alias("ICAO24_HEX"),
        pl.col("typecode").alias("ICAO_OTHER")
    )
)
mapping_df.head().collect()

ICAO24_HEX,ICAO_OTHER
str,str
"""aa3487""","""BE36"""
"""a4fa61""","""PA31"""
"""a7a809""","""AC90"""
"""391927""","""DR40"""
"""503c21""","""ZZZZ"""


In [15]:
flight_and_type = flights.join(mapping_df, on="ICAO24_HEX", how="left")
flight_and_type.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str
"""7c6de2""",2025-10-23 08:49:20 AEDT,2025-10-23 09:59:43 AEDT,1h 10m 23s,"""YSSY""","""YMML""","""B738"""
"""7c6d33""",2025-10-23 08:47:50 AEDT,2025-10-23 09:57:31 AEDT,1h 9m 41s,"""YSSY""","""YMML""","""B738"""
"""7c1476""",2025-10-23 08:38:39 AEDT,2025-10-23 09:51:40 AEDT,1h 13m 1s,"""YSSY""","""YMML""","""A332"""
"""7c7cce""",2025-10-23 08:27:15 AEDT,2025-10-23 09:42:23 AEDT,1h 15m 8s,"""YSSY""","""YMML""","""A320"""
"""7c7a48""",2025-10-23 08:15:30 AEDT,2025-10-23 09:33:45 AEDT,1h 18m 15s,"""YSSY""","""YMML""","""B738"""


In [16]:
# from here: https://www.eea.europa.eu/publications/emep-eea-guidebook-2023/part-b-sectoral-guidance-chapters/1-energy/1-a-combustion/1-a-3-a-aviation.3/view
url = "https://www.eea.europa.eu/publications/emep-eea-guidebook-2023/part-b-sectoral-guidance-chapters/1-energy/1-a-combustion/1-a-3-a-aviation.3/at_download/file"
download_file(url, emissions_spreadsheet)

In [17]:
# This sheet in the file is hidden in Excel
# Extract it manually.
(
    pl.read_excel(
        source=emissions_spreadsheet,
        sheet_name="database",
        read_options={
            "skip_rows": 1,
            "header_row": True,
        },
        schema_overrides={"FORECAST  DURATION": pl.Duration()},
    )
    .write_excel(os.path.join(data_dir, "spreadsheet-extracted.xlsx"))
)

In [18]:
# load the emissions spreadsheet
emissions_lookup = (
    pl.read_excel(
        source=emissions_spreadsheet,
        sheet_name="database",
        read_options={
            "skip_rows": 1,
            "header_row": True,
        },
        schema_overrides={"FORECAST  DURATION": pl.Duration()},
    )
    .lazy()
    .filter(pl.col("ICAO_ID").is_not_null())
    .filter(pl.col("ICAO_ID") != "")
    .rename(
        {
            "ICAO_ID": "ICAO_OTHER",
            "AIRCRAFT ID": "AIRCRAFT_ID",
            # watch out, there are double spaces and trailing spaces
            # in the raw data
            "FORECAST  DURATION": "DURATION_REFERENCE",
            "FORECAST CO2 (3,15 for JET-A or 3,10 for AvGAS) ": "CO2",
            "FORECAST  NOX": "NOX",
            "FORECAST  SOX": "SOX",
            "FORECAST  H20": "H2O",
            "FORECAST  CO": "CO",
            "FORECAST  HC": "HC",
            " PM Non Volatile": "PM_NON_VOLATILE",
            "PM VOLATILE (all)": "PM_VOLATILE",
            "PM TOTAL": "PM_TOTAL",
        }
    )
)

emissions_cols = [
    "CO2",
    "NOX",
    "SOX",
    "H2O",
    "CO",
    "HC",
    "PM_NON_VOLATILE",
    "PM_VOLATILE",
    "PM_TOTAL",
]

emissions_lookup.head().collect()

ICAO_OTHER,AIRCRAFT_ID,IMPACT ACFT ID,Manufacturer,One of the models associated with this aircraft type,Description,Engine Type,Number of engines,ENGINE ID LTO,LTO or CCD,CRUISE ALT,ADES,DURATION_REFERENCE,FORECAST FUEL BURNT KG,CO2,NOX,SOX,H2O,CO,HC,PM_NON_VOLATILE,PM_VOLATILE,PM_TOTAL
str,str,str,str,str,str,str,str,str,str,i64,i64,duration[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""A124""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",240,200,34m 1s,6430.264058,20255.331782,168.87469,5.401423,7954.237657,8.272752,3.778494,0.22115,0.689774,0.910925
"""A140""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",300,250,40m 42s,7665.318022,24145.75177,202.240391,6.438867,9481.998763,10.541993,4.727702,0.22849,0.873995,1.102485
"""A148""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",320,500,1h 16m 56s,14398.156664,45354.193491,338.863846,12.094451,17810.519114,16.794008,8.565683,0.350165,1.822895,2.17306
"""A19N""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",340,750,1h 53m 50s,21058.965284,66335.740645,475.476091,17.689524,26049.935059,23.35394,12.527249,0.43895,2.823369,3.262319
"""A20N""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",340,1000,2h 30m 19s,27751.271402,87416.504915,609.057052,23.311065,34328.321892,29.31615,16.318223,0.548512,3.792441,4.340953


Now we do the joins.

Some ICAO_OTHER hex values have a mapping to CCD but not LTO. Use that to get the Aircraft ID. Then look up aircraft ID if there's no match based on ICAO_OTHER.

"LTO" is landing and take off (the emissions while at the airport). "CCD" is cruise, control and descent (mid-flight).
I don't know what `LTO2` is, which is sometimes missing. So we just use `LTO`.

In [19]:
# look up takeoff and landing
flight_and_type = (
    flight_and_type
    .join(
        emissions_lookup
        .select("ICAO_OTHER", "AIRCRAFT_ID")
        .unique()
        , on="ICAO_OTHER", how="left"
    )
)
flight_and_type.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER,AIRCRAFT_ID
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str,str
"""7c6de2""",2025-10-23 08:49:20 AEDT,2025-10-23 09:59:43 AEDT,1h 10m 23s,"""YSSY""","""YMML""","""B738""","""A21N-A"""
"""7c6d33""",2025-10-23 08:47:50 AEDT,2025-10-23 09:57:31 AEDT,1h 9m 41s,"""YSSY""","""YMML""","""B738""","""A21N-A"""
"""7c1476""",2025-10-23 08:38:39 AEDT,2025-10-23 09:51:40 AEDT,1h 13m 1s,"""YSSY""","""YMML""","""A332""","""A124-A"""
"""7c7cce""",2025-10-23 08:27:15 AEDT,2025-10-23 09:42:23 AEDT,1h 15m 8s,"""YSSY""","""YMML""","""A320""","""A124-A"""
"""7c7a48""",2025-10-23 08:15:30 AEDT,2025-10-23 09:33:45 AEDT,1h 18m 15s,"""YSSY""","""YMML""","""B738""","""A21N-A"""


In [20]:
# look up takeoff and landing emissions
flight_and_type = (
    flight_and_type
    .join(
        emissions_lookup
        .filter(pl.col("LTO or CCD") == "LTO")
        .select(["AIRCRAFT_ID"] + [pl.col(c).alias(c + "_LTO") for c in emissions_cols])
        , on="AIRCRAFT_ID", how="left"
    )
)
flight_and_type.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER,AIRCRAFT_ID,CO2_LTO,NOX_LTO,SOX_LTO,H2O_LTO,CO_LTO,HC_LTO,PM_NON_VOLATILE_LTO,PM_VOLATILE_LTO,PM_TOTAL_LTO
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""7c6de2""",2025-10-23 08:49:20 AEDT,2025-10-23 09:59:43 AEDT,1h 10m 23s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992
"""7c6d33""",2025-10-23 08:47:50 AEDT,2025-10-23 09:57:31 AEDT,1h 9m 41s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992
"""7c1476""",2025-10-23 08:38:39 AEDT,2025-10-23 09:51:40 AEDT,1h 13m 1s,"""YSSY""","""YMML""","""A332""","""A124-A""",10769.22,67.73848,2.871792,4229.0554,20.070542,3.73421,0.10514,0.244812,0.3499524
"""7c7cce""",2025-10-23 08:27:15 AEDT,2025-10-23 09:42:23 AEDT,1h 15m 8s,"""YSSY""","""YMML""","""A320""","""A124-A""",10769.22,67.73848,2.871792,4229.0554,20.070542,3.73421,0.10514,0.244812,0.3499524
"""7c7a48""",2025-10-23 08:15:30 AEDT,2025-10-23 09:33:45 AEDT,1h 18m 15s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992


Cruise, control and descent (CCD) is harder. Most have many matches, for varying duration/distance. We need to match to two rows, and linearly interpolate in between them.


In [21]:
emissions_options = (
    flight_and_type
    .join(emissions_lookup,
          on="AIRCRAFT_ID",
          how="left"
    )
)

In [22]:
emissions_lower = (
    emissions_options
    .sort("DURATION_REFERENCE", descending=False)
    .filter(pl.col("DURATION_REFERENCE") >= pl.col("FLIGHT_DURATION_ACTUAL"))
    .group_by("AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL")
    .first()
    .select(
        ["AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL", pl.col("DURATION_REFERENCE").alias("DURATION_REFERENCE_LOWER")]
        + [pl.col(c).alias(c + "_LOWER") for c in emissions_cols]
    )
)

emissions_upper = (
    emissions_options
    .sort("DURATION_REFERENCE", descending=True)
    .filter(pl.col("DURATION_REFERENCE") <= pl.col("FLIGHT_DURATION_ACTUAL"))
    .group_by("AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL")
    .first()
    .select(
        ["AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL", pl.col("DURATION_REFERENCE").alias("DURATION_REFERENCE_UPPER")]
        + [pl.col(c).alias(c + "_UPPER") for c in emissions_cols]
    )
)

In [23]:
flight_emissions = (
    flight_and_type
    .join(emissions_lower, on=["AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL"], how="left")
    .join(emissions_upper, on=["AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL"], how="left")
    .with_columns(
        ((pl.col("FLIGHT_DURATION_ACTUAL") - pl.col("DURATION_REFERENCE_LOWER")) / (pl.col("DURATION_REFERENCE_UPPER") - pl.col("DURATION_REFERENCE_LOWER"))).alias("INTERPOLATION_FACTOR")
    )
    .with_columns(
        [
            (pl.col(c + "_LOWER") + pl.col("INTERPOLATION_FACTOR") * (pl.col(c + "_UPPER") - pl.col(c + "_LOWER"))).alias(c + "_CCD")
            for c in emissions_cols
        ]
    )
    .select(
        pl.exclude([c + app for c in emissions_cols for app in ["_UPPER", "_LOWER"]])
    )
)
flight_emissions.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER,AIRCRAFT_ID,CO2_LTO,NOX_LTO,SOX_LTO,H2O_LTO,CO_LTO,HC_LTO,PM_NON_VOLATILE_LTO,PM_VOLATILE_LTO,PM_TOTAL_LTO,DURATION_REFERENCE_LOWER,DURATION_REFERENCE_UPPER,INTERPOLATION_FACTOR,CO2_CCD,NOX_CCD,SOX_CCD,H2O_CCD,CO_CCD,HC_CCD,PM_NON_VOLATILE_CCD,PM_VOLATILE_CCD,PM_TOTAL_CCD
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,duration[μs],duration[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""7c6de2""",2025-10-23 08:49:20 AEDT,2025-10-23 09:59:43 AEDT,1h 10m 23s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 12m 25s,39m 15s,0.061307,8849.156903,43.237943,2.359776,3475.050851,7.778982,0.190012,0.098807,0.226448,0.325255
"""7c6d33""",2025-10-23 08:47:50 AEDT,2025-10-23 09:57:31 AEDT,1h 9m 41s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 12m 25s,39m 15s,0.082412,8764.252822,42.893688,2.337134,3441.709124,7.739016,0.188491,0.098377,0.223946,0.322323
"""7c1476""",2025-10-23 08:38:39 AEDT,2025-10-23 09:51:40 AEDT,1h 13m 1s,"""YSSY""","""YMML""","""A332""","""A124-A""",10769.22,67.73848,2.871792,4229.0554,20.070542,3.73421,0.10514,0.244812,0.3499524,1h 16m 56s,40m 42s,0.108096,43061.652643,324.095441,11.483107,16910.242075,16.118193,8.150814,0.337013,1.720323,2.057335
"""7c7cce""",2025-10-23 08:27:15 AEDT,2025-10-23 09:42:23 AEDT,1h 15m 8s,"""YSSY""","""YMML""","""A320""","""A124-A""",10769.22,67.73848,2.871792,4229.0554,20.070542,3.73421,0.10514,0.244812,0.3499524,1h 16m 56s,40m 42s,0.049678,44300.60025,332.076664,11.813493,17396.774773,16.483421,8.37502,0.34412,1.775755,2.119876
"""7c7a48""",2025-10-23 08:15:30 AEDT,2025-10-23 09:33:45 AEDT,1h 18m 15s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 45m 34s,1h 12m 25s,0.824032,9837.841159,47.291849,2.623425,3863.305608,8.15512,0.206461,0.104466,0.25511,0.359576


In [24]:
# Fill in empty values with the mean of the column
cols_to_fill = [c + app for c in emissions_cols for app in ["_LTO", "_CCD"]]
flight_emissions = (
    flight_emissions
    .with_columns(
        pl.col(cols_to_fill).fill_null(pl.col(cols_to_fill).mean())
    )
)

In [25]:
flight_emissions.collect_schema().keys()

odict_keys(['ICAO24_HEX', 'DEPARTURE_TIME', 'ARRIVAL_TIME', 'FLIGHT_DURATION_ACTUAL', 'DEPARTURE_AIRPORT', 'ARRIVAL_AIRPORT', 'ICAO_OTHER', 'AIRCRAFT_ID', 'CO2_LTO', 'NOX_LTO', 'SOX_LTO', 'H2O_LTO', 'CO_LTO', 'HC_LTO', 'PM_NON_VOLATILE_LTO', 'PM_VOLATILE_LTO', 'PM_TOTAL_LTO', 'DURATION_REFERENCE_LOWER', 'DURATION_REFERENCE_UPPER', 'INTERPOLATION_FACTOR', 'CO2_CCD', 'NOX_CCD', 'SOX_CCD', 'H2O_CCD', 'CO_CCD', 'HC_CCD', 'PM_NON_VOLATILE_CCD', 'PM_VOLATILE_CCD', 'PM_TOTAL_CCD'])

## Sanity Check

What's the total emissions per day?

The CO2 in the raw data is in kg.

In [26]:
(
    flight_emissions
    .group_by(pl.col("DEPARTURE_TIME").dt.date().alias("DEPARTURE_DATE"))
    .agg([
        (pl.col(c + "_LTO").sum() + pl.col(c + "_CCD").sum()).alias(c)
        for c in emissions_cols
    ])
    .sort("DEPARTURE_DATE")
    .collect()
)

DEPARTURE_DATE,CO2,NOX,SOX,H2O,CO,HC,PM_NON_VOLATILE,PM_VOLATILE,PM_TOTAL
date,f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-10-18,1.5871e6,10555.356844,423.228572,623254.470949,1390.305703,275.66161,14.776573,52.135511,66.912084
2025-10-19,2.6513e6,16696.307946,707.0047,1.0411e6,2553.140338,388.988892,25.785783,82.87516,108.660943
2025-10-20,2.9706e6,18432.923531,792.165311,1.1666e6,2924.883338,411.878426,31.023061,91.595484,122.618545
2025-10-21,2.7508e6,17276.132304,733.558018,1.0803e6,2673.36365,399.061777,26.313021,85.623406,111.936427
2025-10-22,2.7158e6,16515.835519,724.200912,1.0665e6,2636.572773,358.508403,27.125804,83.712932,110.838736
2025-10-23,838929.085875,5058.960701,223.714429,329446.144205,870.441584,106.214115,7.949921,25.130413,33.080334


So there's around 2.5 thousand tonnes of CO2 burnt per day.

Is that the right ballpark?

[Google Flights](https://www.google.com/travel/flights/search?tfs=CBwQAholEgoyMDI1LTExLTEyKABqDAgDEggvbS8wNnk1N3IHCAESA01FTEABSAFwAYIBCwj___________8BmAEC) says per-passenger emissions are around 77kg on average.

How many flights? Google Flights says 34 + 24 + 13 = 71 from Syd to Melb on 12/11/2025. Double that, it's 142 each way. [This source](https://www.thenewdaily.com.au/travel/2019/03/27/busiest-flight-routes-australia) says almost 150.

How many seats per flight? Google Flights assumes economy seats. How do they allocate emissions between classes? Maybe 160 passengers ([Wikipedia](https://en.wikipedia.org/wiki/Airbus_A320_family)).

In [27]:
flights_per_day = 71 * 2
passengers_per_flight = 160
emissions_per_passenger = 77

emissions_per_day = emissions_per_passenger * passengers_per_flight * flights_per_day
emissions_per_day

1749440

Ok, so we're in the right ballpark.

## Airport Data

Add data about the airports to each flight

In [28]:
airport_locations = pl.DataFrame([
    {
        "AIRPORT_ID": v[airport_id_type],
        "LATITUDE": v['location']['latitude'],
        "LONGITUDE": v['location']['longitude'],
    }
    for v in airport_info.values()
])
airport_locations

AIRPORT_ID,LATITUDE,LONGITUDE
str,f64,f64
"""YMML""",-37.673333,144.843333
"""YSSY""",-33.946111,151.177222


In [29]:
flight_emissions = (
    flight_emissions
    .join(
        airport_locations
            .lazy()
            .rename({"AIRPORT_ID": "DEPARTURE_AIRPORT", 
                                   "LATITUDE": "LATITUDE_DEPARTURE",
                                   "LONGITUDE": "LONGITUDE_DEPARTURE"}),
        on="DEPARTURE_AIRPORT"
    )
    .join(
        airport_locations
            .lazy()
            .rename({"AIRPORT_ID": "ARRIVAL_AIRPORT",
                                   "LATITUDE": "LATITUDE_ARRIVAL", 
                                   "LONGITUDE": "LONGITUDE_ARRIVAL"}),
        on="ARRIVAL_AIRPORT"
    )
    # calculate the angle
    # 0 is east, 90 is north
    # Use arctan2 to handle all quadrants correctly
    .with_columns(
        (pl.col("LATITUDE_ARRIVAL") - pl.col("LATITUDE_DEPARTURE")).alias("LATITUDE_DELTA"),
        (pl.col("LONGITUDE_ARRIVAL") - pl.col("LONGITUDE_DEPARTURE")).alias("LONGITUDE_DELTA"),
    )
    .with_columns(
        pl.arctan2(pl.col("LATITUDE_DELTA"), pl.col("LONGITUDE_DELTA")).degrees().alias("ANGLE")
    )
)

In [30]:
flight_emissions.select("ARRIVAL_AIRPORT", "DEPARTURE_AIRPORT", "ANGLE").unique().head().collect()

ARRIVAL_AIRPORT,DEPARTURE_AIRPORT,ANGLE
str,str,f64
"""YSSY""","""YMML""",30.474986
"""YMML""","""YSSY""",-149.525014


## Mid-Flight Data

We generate a list of timestamps which we care about.

OpenSky doesn't offer mid-flight positions for historcal flights, so we'll just interpolate.

In [35]:
# choose the second date
# (first might be partial day because of timezone boundaries)
_date = (
    flight_emissions
    .select(pl.col("DEPARTURE_TIME").dt.date().alias("DATE"))
    .unique()
    .sort("DATE")
    .head(2)
    .tail(1)
    .collect()
    .item()
)
_date

datetime.date(2025, 10, 19)

In [76]:
num_slices = 24 * 60

times_lf = pl.LazyFrame({
    "TIME": pl.datetime_range(
        start=dt.datetime.combine(_date, dt.time.min, tzinfo=sydney_tz),
        end=dt.datetime.combine(_date + dt.timedelta(days=1), dt.time.min, tzinfo=sydney_tz),
        interval=dt.timedelta(days=1) / (num_slices),
        closed='both',
        eager=True
    )
})
    
times_lf.collect()

TIME
"datetime[μs, Australia/Sydney]"
2025-10-19 00:00:00 AEDT
2025-10-19 00:01:00 AEDT
2025-10-19 00:02:00 AEDT
2025-10-19 00:03:00 AEDT
2025-10-19 00:04:00 AEDT
…
2025-10-19 23:56:00 AEDT
2025-10-19 23:57:00 AEDT
2025-10-19 23:58:00 AEDT


In [77]:
jitter_scale_end = 0.1
jitter_scale_mid = 1.5

# returns a random value [0,1] , different for each row
def random_col(scale=jitter_scale_end) -> pl.Expr:
    return (pl.int_range(pl.len()).map_batches(lambda s: pl.Series(np.random.random(len(s))), return_dtype=pl.Float64) - 0.5) * 2 * scale

animation_data = (
    flight_emissions
    .with_columns(
        random_col().alias("JITTER_LATITUDE_DEPARTURE"),
        random_col().alias("JITTER_LONGITUDE_DEPARTURE"),
        random_col().alias("JITTER_LATITUDE_ARRIVAL"),
        random_col().alias("JITTER_LONGITUDE_ARRIVAL"),
        random_col(jitter_scale_mid).alias("JITTER_LATITUDE_MID"),
        random_col(jitter_scale_mid).alias("JITTER_LONGITUDE_MID"),
    )
    .with_columns(
        (pl.col("LONGITUDE_DEPARTURE") + pl.col("JITTER_LONGITUDE_DEPARTURE")).alias("LONGITUDE_DEPARTURE"),
        (pl.col("LATITUDE_DEPARTURE") + pl.col("JITTER_LATITUDE_DEPARTURE")).alias("LATITUDE_DEPARTURE"),
        (pl.col("LONGITUDE_ARRIVAL") + pl.col("JITTER_LONGITUDE_ARRIVAL")).alias("LONGITUDE_ARRIVAL"),
        (pl.col("LATITUDE_ARRIVAL") + pl.col("JITTER_LATITUDE_ARRIVAL")).alias("LATITUDE_ARRIVAL"),
    )
    .join(times_lf, how="cross")
    .filter(pl.col("DEPARTURE_TIME") >= pl.col("TIME").min())
    .with_columns(
        (pl.col("TIME") >= pl.col("DEPARTURE_TIME")).alias("HAS_DEPARTED"),
        (pl.col("TIME") >= pl.col("ARRIVAL_TIME")).alias("HAS_ARRIVED"),
        pl.col("TIME").is_between("DEPARTURE_TIME", "ARRIVAL_TIME").alias("IN_AIR"),
        ((pl.col("TIME") - pl.col("DEPARTURE_TIME")) / (pl.col("ARRIVAL_TIME") - pl.col("DEPARTURE_TIME")))
        .clip(lower_bound=0, upper_bound=2)
        .alias("FLIGHT_PROGRESS")
    )
    .with_columns([
        (
            pl.col("FLIGHT_PROGRESS") * pl.col(c + "_CCD")
            + pl.col("HAS_DEPARTED") * pl.col(c + "_LTO")
        ).alias(c)
        for c in emissions_cols
    ])
    .with_columns(
        (
            pl.col("LONGITUDE_DEPARTURE") + 
            pl.col("FLIGHT_PROGRESS") * (pl.col("LONGITUDE_ARRIVAL") - pl.col("LONGITUDE_DEPARTURE")) +
            pl.col("JITTER_LATITUDE_MID") * pl.col("FLIGHT_PROGRESS") * (1 - pl.col("FLIGHT_PROGRESS"))
        ).alias("LONGITUDE"),
        (
            pl.col("LATITUDE_DEPARTURE") + 
            pl.col("FLIGHT_PROGRESS") * (pl.col("LATITUDE_ARRIVAL") - pl.col("LATITUDE_DEPARTURE")) +
            pl.col("JITTER_LONGITUDE_MID") * pl.col("FLIGHT_PROGRESS") * (1 - pl.col("FLIGHT_PROGRESS"))
        ).alias("LATITUDE"),
    )
    .sort("TIME", "DEPARTURE_TIME", "ICAO24_HEX")
)
animation_data.collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER,AIRCRAFT_ID,CO2_LTO,NOX_LTO,SOX_LTO,H2O_LTO,CO_LTO,HC_LTO,PM_NON_VOLATILE_LTO,PM_VOLATILE_LTO,PM_TOTAL_LTO,DURATION_REFERENCE_LOWER,DURATION_REFERENCE_UPPER,INTERPOLATION_FACTOR,CO2_CCD,NOX_CCD,SOX_CCD,H2O_CCD,CO_CCD,HC_CCD,PM_NON_VOLATILE_CCD,PM_VOLATILE_CCD,PM_TOTAL_CCD,LATITUDE_DEPARTURE,LONGITUDE_DEPARTURE,LATITUDE_ARRIVAL,LONGITUDE_ARRIVAL,LATITUDE_DELTA,LONGITUDE_DELTA,ANGLE,JITTER_LATITUDE_DEPARTURE,JITTER_LONGITUDE_DEPARTURE,JITTER_LATITUDE_ARRIVAL,JITTER_LONGITUDE_ARRIVAL,JITTER_LATITUDE_MID,JITTER_LONGITUDE_MID,TIME,HAS_DEPARTED,HAS_ARRIVED,IN_AIR,FLIGHT_PROGRESS,CO2,NOX,SOX,H2O,CO,HC,PM_NON_VOLATILE,PM_VOLATILE,PM_TOTAL,LONGITUDE,LATITUDE
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,duration[μs],duration[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,"datetime[μs, Australia/Sydney]",bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""7c6b09""",2025-10-19 06:07:36 AEDT,2025-10-19 07:11:59 AEDT,1h 4m 23s,"""YMML""","""YSSY""","""A320""","""A124-A""",10769.22,67.73848,2.871792,4229.0554,20.070542,3.73421,0.10514,0.244812,0.3499524,1h 16m 56s,40m 42s,0.346366,38008.307283,291.542106,10.135548,14925.801623,14.628522,7.236336,0.308021,1.494228,1.802249,-37.671623,144.791667,-33.87479,151.158325,3.727222,6.333889,30.474986,0.00171,-0.051666,0.071321,-0.018897,-0.795895,-0.61401,2025-10-19 00:00:00 AEDT,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.791667,-37.671623
"""7c6c96""",2025-10-19 06:22:03 AEDT,2025-10-19 07:36:44 AEDT,1h 14m 41s,"""YSSY""","""YMML""","""A320""","""A124-A""",10769.22,67.73848,2.871792,4229.0554,20.070542,3.73421,0.10514,0.244812,0.3499524,1h 16m 56s,40m 42s,0.062098,44037.20194,330.379869,11.743253,17293.338687,16.405774,8.327354,0.342609,1.763971,2.10658,-33.951387,151.263518,-37.70586,144.926265,-3.727222,-6.333889,-149.525014,-0.005276,0.086296,-0.032527,0.082932,0.434866,-1.46943,2025-10-19 00:00:00 AEDT,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,151.263518,-33.951387
"""7c7800""",2025-10-19 06:22:35 AEDT,2025-10-19 07:25:55 AEDT,1h 3m 20s,"""YMML""","""YSSY""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 12m 25s,39m 15s,0.273869,7994.05151,39.770796,2.131747,3139.252033,7.376471,0.17469,0.094478,0.201246,0.295724,-37.695827,144.885875,-34.015559,151.149561,3.727222,6.333889,30.474986,-0.022494,0.042542,-0.069448,-0.027661,0.520271,1.497072,2025-10-19 00:00:00 AEDT,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.885875,-37.695827
"""7c6db6""",2025-10-19 06:24:42 AEDT,2025-10-19 07:49:05 AEDT,1h 24m 23s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 45m 34s,1h 12m 25s,0.639015,10618.062262,50.502832,2.831483,4169.69713,8.428541,0.219109,0.109103,0.277604,0.386707,-33.887764,151.207419,-37.716179,144.857817,-3.727222,-6.333889,-149.525014,0.058347,0.030197,-0.042846,0.014484,0.502831,-0.901789,2025-10-19 00:00:00 AEDT,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,151.207419,-33.887764
"""7c6d7f""",2025-10-19 07:01:15 AEDT,2025-10-19 08:15:50 AEDT,1h 14m 35s,"""YSSY""","""YMML""","""A321""","""A124-A""",10769.22,67.73848,2.871792,4229.0554,20.070542,3.73421,0.10514,0.244812,0.3499524,1h 16m 56s,40m 42s,0.064857,43978.668982,330.002803,11.727644,17270.352891,16.388519,8.316761,0.342274,1.761352,2.103625,-34.028183,151.20302,-37.764633,144.934277,-3.727222,-6.333889,-149.525014,-0.082072,0.025798,-0.0913,0.090944,-0.366307,1.025043,2025-10-19 00:00:00 AEDT,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,151.20302,-34.028183
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…

`animation_data` has one row per (flight, time). The emissions values are cumulative, within each flight. So for total emissions, group by time, sum across flights.

In [78]:
os.makedirs(results_dir, exist_ok=True)

In [79]:
(
    animation_data
    # .filter(pl.col("TIME") == pl.col("TIME").max())
    .filter(pl.col("DEPARTURE_TIME") <= pl.col("TIME"))
    .filter(pl.col("ARRIVAL_TIME") >= pl.col("TIME"))
    .sort("TIME", "DEPARTURE_TIME", "ARRIVAL_TIME")
    .select("TIME", "DEPARTURE_TIME", "ARRIVAL_TIME", "ICAO24_HEX")
    .tail()
    .collect()
)

TIME,DEPARTURE_TIME,ARRIVAL_TIME,ICAO24_HEX
"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",str
2025-10-19 23:29:00 AEDT,2025-10-19 22:18:40 AEDT,2025-10-19 23:33:02 AEDT,"""7c1472"""
2025-10-19 23:30:00 AEDT,2025-10-19 22:18:40 AEDT,2025-10-19 23:33:02 AEDT,"""7c1472"""
2025-10-19 23:31:00 AEDT,2025-10-19 22:18:40 AEDT,2025-10-19 23:33:02 AEDT,"""7c1472"""
2025-10-19 23:32:00 AEDT,2025-10-19 22:18:40 AEDT,2025-10-19 23:33:02 AEDT,"""7c1472"""
2025-10-19 23:33:00 AEDT,2025-10-19 22:18:40 AEDT,2025-10-19 23:33:02 AEDT,"""7c1472"""


In [80]:
(
    animation_data
    .group_by("TIME")
    .agg([
        pl.col(c).sum()
        for c in emissions_cols
    ] + [
        pl.struct("ICAO24_HEX", "DEPARTURE_TIME", "ARRIVAL_TIME").filter("HAS_DEPARTED").n_unique().alias("NUM_FLIGHTS")
    ])
    # # reset counter to zero (some flights prior to the start time were counted)
    # .sort("TIME")
    # .with_columns(
    #     (pl.exclude("TIME") - pl.exclude("TIME").first())
    # )
    .sort("TIME")
    .sink_parquet(emissions_results_path)
)
animation_emissions_data = pl.read_parquet(emissions_results_path)
animation_emissions_data

TIME,CO2,NOX,SOX,H2O,CO,HC,PM_NON_VOLATILE,PM_VOLATILE,PM_TOTAL,NUM_FLIGHTS
"datetime[μs, Australia/Sydney]",f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
2025-10-19 00:00:00 AEDT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2025-10-19 00:01:00 AEDT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2025-10-19 00:02:00 AEDT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2025-10-19 00:03:00 AEDT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2025-10-19 00:04:00 AEDT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
…,…,…,…,…,…,…,…,…,…,…
2025-10-19 23:56:00 AEDT,4.7076e6,30108.405825,1255.352976,1.8487e6,3728.224725,643.189325,44.763189,153.046527,197.809716,121
2025-10-19 23:57:00 AEDT,4.7083e6,30113.719347,1255.555618,1.8490e6,3728.57032,643.308862,44.769495,153.074935,197.84443,121
2025-10-19 23:58:00 AEDT,4.7091e6,30119.032869,1255.75826,1.8493e6,3728.915916,643.428398,44.775801,153.103343,197.879144,121


In [81]:
(
    animation_data
    .sort("ICAO24_HEX", "DEPARTURE_TIME", "TIME")
    .select("TIME", "LATITUDE", "LONGITUDE", "ANGLE", "IN_AIR")
    .sink_parquet(planes_results_path)
)
animation_plane_data = pl.read_parquet(planes_results_path)
animation_plane_data

TIME,LATITUDE,LONGITUDE,ANGLE,IN_AIR
"datetime[μs, Australia/Sydney]",f64,f64,f64,bool
2025-10-19 00:00:00 AEDT,-33.951836,151.119762,-149.525014,false
2025-10-19 00:01:00 AEDT,-33.951836,151.119762,-149.525014,false
2025-10-19 00:02:00 AEDT,-33.951836,151.119762,-149.525014,false
2025-10-19 00:03:00 AEDT,-33.951836,151.119762,-149.525014,false
2025-10-19 00:04:00 AEDT,-33.951836,151.119762,-149.525014,false
…,…,…,…,…
2025-10-19 23:56:00 AEDT,-33.92471,151.257213,-149.525014,false
2025-10-19 23:57:00 AEDT,-33.92471,151.257213,-149.525014,false
2025-10-19 23:58:00 AEDT,-33.92471,151.257213,-149.525014,false
